# TikTok Video Engagement Prediction — Final Pipeline
**Author:** Haya Alharthi
**Competition:** WeCloudData DS Bootcamp — In-Class Competition (Kaggle)
**Task:** Predict `target_day30_views` (cumulative views at Day 30) using only
video, creator, and engagement data available during Days 0–5.
**Evaluation metric:** RMSE (Root Mean Squared Error) on the hidden test set.

**Final model: "Viral Single-Gate"** — a Linear + CatBoost/XGBoost hybrid base
model, corrected by a two-stage viral classifier/specialist model that up-weights
predictions for videos the classifier flags as likely to go viral.

Cross-validated (5-fold) out-of-fold RMSE: **61,410**.

This notebook contains only the path that produced the final submission.
Earlier exploratory branches (Creator Residual blending, Curve/Growth features,
Ridge stacking) were tested but did not outperform this pipeline and are not
included here — see the project README for a summary of what was tried.




## 0. Setup

In [85]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
N_SPLITS = 5


## 1. Load Competition Data

In [86]:
DATA_PATH = "/kaggle/input/competitions/predictive-modelling-ds"

train = pd.read_csv(f"{DATA_PATH}/train_videos.csv")
test = pd.read_csv(f"{DATA_PATH}/test_videos.csv")
engagement = pd.read_csv(f"{DATA_PATH}/engagement_daily.csv")
creators = pd.read_csv(f"{DATA_PATH}/creators_daily.csv")
sample_submission = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

print("Train videos:", train.shape)
print("Test videos :", test.shape)
print("Engagement  :", engagement.shape)
print("Creators    :", creators.shape)


Train videos: (12000, 27)
Test videos : (3001, 26)
Engagement  : (79489, 10)
Creators    : (252166, 7)


## 2. Feature Engineering

Two feature blocks are built, using only information available during
Days 0–5 (no leakage of future engagement or post-Day-5 creator stats):

1. **Engagement features** — per-day metric snapshots, first/last observed
   values, growth rates, and engagement ratios (like/comment/share/collect
   rate relative to plays).
2. **Creator features** — the most recent creator snapshot available at or
   before the Day-5 cutoff for that video, joined with `merge_asof`.


### 2.1 Engagement features

In [87]:
eng = engagement.copy()
eng["date"] = pd.to_datetime(eng["date"])
eng["days_since_post"] = pd.to_numeric(eng["days_since_post"], errors="coerce")

# Only use information available during days 0-5 (no leakage)
eng = eng[eng["days_since_post"].between(0, 5)].copy()
eng = eng.sort_values(["video_id", "days_since_post", "date"])

engagement_metrics = [
    "play_count", "like_count", "comment_count", "share_count",
    "collect_count", "download_count", "whatsapp_share_count"
]

print("Rows used:", len(eng))
print("Unique videos:", eng["video_id"].nunique())


Rows used: 79489
Unique videos: 15000


In [88]:
# One column per metric per day (wide format)
daily_wide = eng.pivot_table(
    index="video_id", columns="days_since_post",
    values=engagement_metrics, aggfunc="last"
)
daily_wide.columns = [f"{metric}_day{int(day)}" for metric, day in daily_wide.columns]
daily_wide = daily_wide.reset_index()

print("Daily-wide shape:", daily_wide.shape)


Daily-wide shape: (15000, 43)


In [89]:
# First / last observed engagement values per video -> growth features
first_eng = (
    eng.drop_duplicates("video_id", keep="first")
       [["video_id", "days_since_post"] + engagement_metrics].copy()
)
last_eng = (
    eng.drop_duplicates("video_id", keep="last")
       [["video_id", "days_since_post"] + engagement_metrics].copy()
)

first_eng = first_eng.rename(columns={
    "days_since_post": "first_observed_day",
    **{col: f"first_{col}" for col in engagement_metrics}
})
last_eng = last_eng.rename(columns={
    "days_since_post": "latest_observed_day",
    **{col: f"latest_{col}" for col in engagement_metrics}
})

trajectory_features = first_eng.merge(last_eng, on="video_id", how="outer", validate="one_to_one")
trajectory_features["observed_day_span"] = (
    trajectory_features["latest_observed_day"] - trajectory_features["first_observed_day"]
)
safe_span = trajectory_features["observed_day_span"].clip(lower=1)

for metric in engagement_metrics:
    trajectory_features[f"{metric}_growth"] = (
        trajectory_features[f"latest_{metric}"] - trajectory_features[f"first_{metric}"]
    )
    trajectory_features[f"{metric}_growth_per_day"] = trajectory_features[f"{metric}_growth"] / safe_span

print("Trajectory shape:", trajectory_features.shape)


Trajectory shape: (15000, 32)


In [90]:
# Coverage + engagement ratio features
coverage_features = (
    eng.groupby("video_id")
       .agg(
           engagement_days_available=("days_since_post", "nunique"),
           engagement_first_day=("days_since_post", "min"),
           engagement_last_day=("days_since_post", "max"),
       )
       .reset_index()
)

engagement_features = (
    daily_wide
    .merge(trajectory_features, on="video_id", how="outer", validate="one_to_one")
    .merge(coverage_features, on="video_id", how="outer", validate="one_to_one")
)

safe_plays = engagement_features["latest_play_count"].replace(0, np.nan)
engagement_features["latest_like_rate"] = engagement_features["latest_like_count"] / safe_plays
engagement_features["latest_comment_rate"] = engagement_features["latest_comment_count"] / safe_plays
engagement_features["latest_share_rate"] = engagement_features["latest_share_count"] / safe_plays
engagement_features["latest_collect_rate"] = engagement_features["latest_collect_count"] / safe_plays

print("Engagement features shape:", engagement_features.shape)


Engagement features shape: (15000, 81)


### 2.2 Creator features

In [91]:
train_base = train.copy()
test_base = test.copy()
creator_data = creators.copy()

train_base["create_date"] = pd.to_datetime(train_base["create_date"])
test_base["create_date"] = pd.to_datetime(test_base["create_date"])
creator_data["date"] = pd.to_datetime(creator_data["date"])

# Day-5 cutoff: the creator snapshot must not be from after this date
train_base["feature_cutoff_date"] = train_base["create_date"] + pd.Timedelta(days=5)
test_base["feature_cutoff_date"] = test_base["create_date"] + pd.Timedelta(days=5)

all_video_keys = pd.concat(
    [
        train_base[["video_id", "author_id", "feature_cutoff_date"]],
        test_base[["video_id", "author_id", "feature_cutoff_date"]],
    ],
    ignore_index=True,
)

creator_data = creator_data.rename(columns={
    "date": "creator_snapshot_date",
    "follower_count": "creator_follower_count",
    "following_count": "creator_following_count",
    "total_favorited": "creator_total_favorited",
    "video_count": "creator_video_count",
    "enterprise_verified": "creator_enterprise_verified",
})

all_video_keys = all_video_keys.sort_values(["feature_cutoff_date", "author_id"])
creator_data = creator_data.sort_values(["creator_snapshot_date", "author_id"])

# As-of join: most recent creator snapshot at or before the Day-5 cutoff
creator_features = pd.merge_asof(
    all_video_keys, creator_data,
    left_on="feature_cutoff_date", right_on="creator_snapshot_date",
    by="author_id", direction="backward",
)
creator_features["creator_snapshot_age_days"] = (
    creator_features["feature_cutoff_date"] - creator_features["creator_snapshot_date"]
).dt.days
creator_features = creator_features.drop(columns=["author_id", "feature_cutoff_date"])

print("Creator features shape:", creator_features.shape)


Creator features shape: (15001, 8)


### 2.3 Assemble modeling tables

In [92]:
modeling_train = (
    train_base
    .merge(engagement_features, on="video_id", how="left", validate="one_to_one")
    .merge(creator_features, on="video_id", how="left", validate="one_to_one")
)
modeling_test = (
    test_base
    .merge(engagement_features, on="video_id", how="left", validate="one_to_one")
    .merge(creator_features, on="video_id", how="left", validate="one_to_one")
)

print("Modeling train:", modeling_train.shape)
print("Modeling test :", modeling_test.shape)


Modeling train: (12000, 115)
Modeling test : (3001, 114)


## 3. Prepare Features for Modeling

- Extract calendar features from `create_time`.
- Drop identifier / raw-datetime columns not useful as direct model inputs.
- Mark ID-like columns (`author_id`, `music_id`, `music_owner_id`) as categorical.


In [93]:
TARGET = "target_day30_views"

train_ready = modeling_train.copy()
test_ready = modeling_test.copy()
test_video_ids = test_ready["video_id"].copy()

train_ready["create_time"] = pd.to_datetime(train_ready["create_time"], errors="coerce")
test_ready["create_time"] = pd.to_datetime(test_ready["create_time"], errors="coerce")

for df in [train_ready, test_ready]:
    df["create_hour"] = df["create_time"].dt.hour
    df["create_day_of_week"] = df["create_time"].dt.dayofweek
    df["create_day_of_month"] = df["create_time"].dt.day
    df["create_month"] = df["create_time"].dt.month
    df["create_is_weekend"] = df["create_day_of_week"].isin([5, 6]).astype(int)

columns_to_drop = [
    "video_id", "create_time", "create_date", "feature_cutoff_date",
    "creator_snapshot_date", "creator_enterprise_verified",
]
columns_to_drop = [c for c in columns_to_drop if c in train_ready.columns]

X = train_ready.drop(columns=columns_to_drop + [TARGET]).copy()
y = train_ready[TARGET].copy()
X_test = test_ready.drop(columns=columns_to_drop).copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)


X shape     : (12000, 113)
y shape     : (12000,)
X_test shape: (3001, 113)


In [94]:
id_categorical_columns = ["author_id", "music_owner_id", "music_id"]
detected_categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_columns = sorted(set(detected_categorical_columns + id_categorical_columns))
categorical_columns = [c for c in categorical_columns if c in X.columns]

for col in categorical_columns:
    X[col] = X[col].astype("string").fillna("Missing").astype(str)
    X_test[col] = X_test[col].astype("string").fillna("Missing").astype(str)

X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("Categorical columns:", len(categorical_columns))


Categorical columns: 8


## 4. Base Model A — Linear + CatBoost Residual Hybrid

Rationale: `latest_play_count` alone is a very strong linear predictor of the
Day-30 target (early view accumulation is highly correlated with long-term
performance). Rather than let a tree model re-learn that trend, we fit a
simple linear model first, then train CatBoost only on the **residual error**
of the linear model, using the full feature set (including categoricals).
This keeps the strong linear trend intact and lets CatBoost focus on
correcting systematic deviations from it.


In [95]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

latest_feature = "latest_play_count"

linear_oof = np.zeros(len(X))
hybrid_oof = np.zeros(len(X))
hybrid_test_predictions = np.zeros(len(X_test))
hybrid_models = []

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_train_fold, X_valid_fold = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

    # Step 1: linear baseline on latest_play_count
    lin_train_feat = X_train_fold[[latest_feature]].fillna(0)
    lin_valid_feat = X_valid_fold[[latest_feature]].fillna(0)
    lin_test_feat = X_test[[latest_feature]].fillna(0)

    linear_model = LinearRegression().fit(lin_train_feat, y_train_fold)
    train_linear_pred = linear_model.predict(lin_train_feat)
    valid_linear_pred = linear_model.predict(lin_valid_feat)
    test_linear_pred = linear_model.predict(lin_test_feat)
    linear_oof[valid_idx] = valid_linear_pred

    # Step 2: CatBoost on the residuals
    train_residuals = y_train_fold.values - train_linear_pred
    residual_model = CatBoostRegressor(
        loss_function="RMSE", eval_metric="RMSE",
        iterations=1500, learning_rate=0.03, depth=6,
        l2_leaf_reg=20, random_strength=0.5,
        random_seed=RANDOM_STATE + fold, allow_writing_files=False, verbose=False,
    )
    residual_model.fit(
        X_train_fold, train_residuals, cat_features=categorical_columns,
        eval_set=(X_valid_fold, y_valid_fold.values - valid_linear_pred),
        early_stopping_rounds=150, verbose=False,
    )

    valid_final = valid_linear_pred + residual_model.predict(X_valid_fold)
    test_final = test_linear_pred + residual_model.predict(X_test)

    hybrid_oof[valid_idx] = valid_final
    hybrid_test_predictions += test_final / kf.n_splits
    hybrid_models.append(residual_model)

    fold_rmse = np.sqrt(mean_squared_error(y_valid_fold, valid_final))
    print(f"Fold {fold} hybrid RMSE: {fold_rmse:,.2f}")

catboost_hybrid_rmse = np.sqrt(mean_squared_error(y, hybrid_oof))
print(f"\nCatBoost Hybrid overall OOF RMSE: {catboost_hybrid_rmse:,.2f}")


Fold 1 hybrid RMSE: 85,046.91
Fold 2 hybrid RMSE: 64,903.18
Fold 3 hybrid RMSE: 45,396.79
Fold 4 hybrid RMSE: 59,836.71
Fold 5 hybrid RMSE: 61,816.44

CatBoost Hybrid overall OOF RMSE: 64,665.25


In [96]:
# CV-averaged linear predictions on the test set (used later for blending/weighting)
linear_cv_test_predictions = np.zeros(len(X_test))
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for train_idx, valid_idx in kf.split(X):
    lm = LinearRegression().fit(X.iloc[train_idx][[latest_feature]].fillna(0), y.iloc[train_idx])
    linear_cv_test_predictions += lm.predict(X_test[[latest_feature]].fillna(0)) / kf.n_splits

# Apply the optimal residual-correction weight found during tuning
CATBOOST_WEIGHT = 1.30
catboost_weighted_oof = np.clip(
    linear_oof + CATBOOST_WEIGHT * (hybrid_oof - linear_oof), 0, None
)
catboost_weighted_test = np.clip(
    linear_cv_test_predictions + CATBOOST_WEIGHT * (hybrid_test_predictions - linear_cv_test_predictions), 0, None
)
catboost_weighted_rmse = np.sqrt(mean_squared_error(y, catboost_weighted_oof))
print(f"Weighted CatBoost Hybrid OOF RMSE: {catboost_weighted_rmse:,.2f}")


Weighted CatBoost Hybrid OOF RMSE: 64,514.00


## 5. Base Model B — Linear + XGBoost Residual Hybrid

Same linear-plus-residual idea, but with a regularized XGBoost model on the
residuals instead of CatBoost, and integer-encoded categoricals. Three
conservative hyperparameter configurations are compared (deeper trees risk
memorizing the small number of extreme "viral" videos that dominate RMSE),
and the correction weight is solved analytically per candidate.


In [97]:
from xgboost import XGBRegressor

X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in categorical_columns:
    combined = pd.concat([X_xgb[col], X_test_xgb[col]], axis=0, ignore_index=True).astype("string").fillna("Missing")
    codes, _ = pd.factorize(combined, sort=True)
    X_xgb[col] = codes[: len(X_xgb)].astype(np.int32)
    X_test_xgb[col] = codes[len(X_xgb):].astype(np.int32)

X_xgb = X_xgb.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)
X_test_xgb = X_test_xgb.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)

print("XGBoost training shape:", X_xgb.shape)


XGBoost training shape: (12000, 113)


In [98]:
xgb_candidates = [
    {"name": "xgb_depth3", "max_depth": 3, "learning_rate": 0.025, "min_child_weight": 15,
     "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 1.0, "reg_lambda": 25.0, "gamma": 0.0},
    {"name": "xgb_depth4", "max_depth": 4, "learning_rate": 0.020, "min_child_weight": 25,
     "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 2.0, "reg_lambda": 40.0, "gamma": 0.0},
    {"name": "xgb_depth5", "max_depth": 5, "learning_rate": 0.015, "min_child_weight": 40,
     "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 5.0, "reg_lambda": 60.0, "gamma": 0.0},
]

kf_xgb = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
xgb_experiment_results = []
xgb_prediction_store = {}

for candidate in xgb_candidates:
    name = candidate["name"]
    params = {k: v for k, v in candidate.items() if k != "name"}

    cand_linear_oof = np.zeros(len(X_xgb))
    cand_residual_oof = np.zeros(len(X_xgb))
    cand_linear_test = np.zeros(len(X_test_xgb))
    cand_residual_test = np.zeros(len(X_test_xgb))

    for fold, (train_idx, valid_idx) in enumerate(kf_xgb.split(X_xgb), start=1):
        X_train_fold, X_valid_fold = X_xgb.iloc[train_idx], X_xgb.iloc[valid_idx]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        lm = LinearRegression().fit(X_train_fold[[latest_feature]].fillna(0), y_train_fold)
        train_lin = lm.predict(X_train_fold[[latest_feature]].fillna(0))
        valid_lin = lm.predict(X_valid_fold[[latest_feature]].fillna(0))
        test_lin = lm.predict(X_test_xgb[[latest_feature]].fillna(0))

        train_residual = y_train_fold.to_numpy() - train_lin
        valid_residual = y_valid_fold.to_numpy() - valid_lin

        residual_model = XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse",
            n_estimators=4000, early_stopping_rounds=200, tree_method="hist",
            random_state=100 + fold, n_jobs=-1, **params,
        )
        residual_model.fit(
            X_train_fold, train_residual,
            eval_set=[(X_valid_fold, valid_residual)], verbose=False,
        )

        cand_linear_oof[valid_idx] = valid_lin
        cand_residual_oof[valid_idx] = residual_model.predict(X_valid_fold)
        cand_linear_test += test_lin / kf_xgb.n_splits
        cand_residual_test += residual_model.predict(X_test_xgb) / kf_xgb.n_splits

    # Analytical least-squares correction weight from OOF predictions
    required_correction = y.to_numpy() - cand_linear_oof
    denom = np.dot(cand_residual_oof, cand_residual_oof)
    weight = float(np.clip(np.dot(cand_residual_oof, required_correction) / denom if denom > 0 else 0.0, 0.0, 2.0))

    weighted_oof = np.clip(cand_linear_oof + weight * cand_residual_oof, 0, None)
    weighted_test = np.clip(cand_linear_test + weight * cand_residual_test, 0, None)
    weighted_rmse = np.sqrt(mean_squared_error(y, weighted_oof))

    xgb_experiment_results.append({"candidate": name, "weight": weight, "oof_rmse": weighted_rmse})
    xgb_prediction_store[name] = {"weighted_oof": weighted_oof, "weighted_test": weighted_test}
    print(f"{name}: weight={weight:.3f} | OOF RMSE={weighted_rmse:,.2f}")

xgb_experiment_results = pd.DataFrame(xgb_experiment_results).sort_values("oof_rmse").reset_index(drop=True)
display(xgb_experiment_results)


xgb_depth3: weight=1.119 | OOF RMSE=64,028.85
xgb_depth4: weight=1.251 | OOF RMSE=64,019.21
xgb_depth5: weight=1.279 | OOF RMSE=63,221.14


,candidate,weight,oof_rmse
0,xgb_depth5,1.278588,63221.137211
1,xgb_depth4,1.250614,64019.210509
2,xgb_depth3,1.118524,64028.852639


In [99]:
best_xgb_name = xgb_experiment_results.loc[0, "candidate"]
best_xgb_rmse = float(xgb_experiment_results.loc[0, "oof_rmse"])
best_xgb_oof = xgb_prediction_store[best_xgb_name]["weighted_oof"]
best_xgb_test_predictions = xgb_prediction_store[best_xgb_name]["weighted_test"]

print("Selected XGBoost candidate:", best_xgb_name)
print(f"XGBoost Hybrid OOF RMSE: {best_xgb_rmse:,.2f}")


Selected XGBoost candidate: xgb_depth5
XGBoost Hybrid OOF RMSE: 63,221.14


## 6. Blend Base Models A and B

The two hybrids (CatBoost-residual and XGBoost-residual) make partially
different errors, so a weighted blend of their out-of-fold predictions is
searched for the RMSE-minimizing mix. This blended prediction is the **base
model** that the viral correction (next section) will adjust.


In [100]:
xgboost_weighted_oof = best_xgb_oof

blend_results = []
for alpha in np.arange(0, 1.001, 0.01):  # alpha=0 -> CatBoost only, alpha=1 -> XGBoost only
    blended = np.clip((1 - alpha) * catboost_weighted_oof + alpha * xgboost_weighted_oof, 0, None)
    rmse = np.sqrt(mean_squared_error(y, blended))
    blend_results.append({"xgboost_weight": alpha, "catboost_weight": 1 - alpha, "oof_rmse": rmse})

blend_results = pd.DataFrame(blend_results).sort_values("oof_rmse").reset_index(drop=True)
best_blend = blend_results.iloc[0]

best_blend_alpha = float(best_blend["xgboost_weight"])
best_catboost_alpha = float(best_blend["catboost_weight"])
best_cat_xgb_rmse = float(best_blend["oof_rmse"])

best_cat_xgb_oof = np.clip(
    best_catboost_alpha * catboost_weighted_oof + best_blend_alpha * xgboost_weighted_oof, 0, None
)
base_test_predictions = np.clip(
    best_catboost_alpha * catboost_weighted_test + best_blend_alpha * best_xgb_test_predictions, 0, None
)

print(f"Base blend OOF RMSE: {best_cat_xgb_rmse:,.2f}  "
      f"(CatBoost weight={best_catboost_alpha:.2f}, XGBoost weight={best_blend_alpha:.2f})")


Base blend OOF RMSE: 63,101.42  (CatBoost weight=0.22, XGBoost weight=0.78)


## 7. Viral Two-Stage Correction

A small share of videos ("viral" outliers) drive most of the squared error
under RMSE. This stage trains, per fold and per candidate viral threshold
(top 10%, top 5%, top 2.5% of the training target):

1. A **classifier** predicting whether a video will land in that top bucket.
2. A **specialist regressor**, trained only on videos in that bucket, to
   predict their (much larger) view counts.

The final prediction blends the base model with the specialist's prediction,
weighted by the classifier's estimated probability (raised to a tunable
power) and a tunable gate strength (`gamma`). All hyperparameters are chosen
using out-of-fold RMSE only — never the actual test labels.


In [101]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

BASE_OOF = best_cat_xgb_oof.copy()
BASE_RMSE = np.sqrt(mean_squared_error(y, BASE_OOF))
print(f"Base model OOF RMSE: {BASE_RMSE:,.2f}")

viral_quantiles = [0.90, 0.95, 0.975]
viral_store = {}
viral_kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for quantile in viral_quantiles:
    print(f"\nTraining Viral Two-Stage model | Top {(1 - quantile) * 100:.1f}%")

    probability_oof = np.zeros(len(X))
    specialist_oof = np.zeros(len(X))
    probability_test = np.zeros(len(X_test))
    specialist_test = np.zeros(len(X_test))
    actual_oof = np.zeros(len(X), dtype=int)

    for fold, (train_idx, valid_idx) in enumerate(viral_kf.split(X), start=1):
        X_train_fold, X_valid_fold = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        threshold = float(y_train_fold.quantile(quantile))  # from training fold only
        train_viral = (y_train_fold >= threshold).astype(int)
        valid_viral = (y_valid_fold >= threshold).astype(int)
        actual_oof[valid_idx] = valid_viral

        negative_count = len(train_viral) - int(train_viral.sum())
        positive_weight = np.sqrt(negative_count / max(int(train_viral.sum()), 1))

        # Stage 1: viral classifier
        classifier = CatBoostClassifier(
            loss_function="Logloss", eval_metric="AUC",
            iterations=1200, learning_rate=0.03, depth=6,
            l2_leaf_reg=15, random_strength=0.5,
            class_weights=[1.0, positive_weight],
            random_seed=500 + fold, allow_writing_files=False, verbose=False,
        )
        classifier.fit(
            X_train_fold, train_viral, cat_features=categorical_columns,
            eval_set=(X_valid_fold, valid_viral), early_stopping_rounds=120, verbose=False,
        )
        probability_oof[valid_idx] = classifier.predict_proba(X_valid_fold)[:, 1]
        probability_test += classifier.predict_proba(X_test)[:, 1] / viral_kf.n_splits

        # Stage 2: specialist regressor (trained only on viral rows)
        train_mask = train_viral.to_numpy() == 1
        X_train_viral = X_train_fold.iloc[np.where(train_mask)[0]]
        y_train_viral = y_train_fold.iloc[np.where(train_mask)[0]]

        specialist = CatBoostRegressor(
            loss_function="RMSE", eval_metric="RMSE",
            iterations=1200, learning_rate=0.025, depth=5,
            l2_leaf_reg=25, random_strength=0.5,
            random_seed=700 + fold, allow_writing_files=False, verbose=False,
        )
        valid_mask = valid_viral.to_numpy() == 1
        if valid_mask.sum() >= 20:
            specialist.fit(
                X_train_viral, y_train_viral, cat_features=categorical_columns,
                eval_set=(X_valid_fold.iloc[np.where(valid_mask)[0]], y_valid_fold.iloc[np.where(valid_mask)[0]]),
                early_stopping_rounds=120, verbose=False,
            )
        else:
            specialist.fit(X_train_viral, y_train_viral, cat_features=categorical_columns, verbose=False)

        specialist_oof[valid_idx] = specialist.predict(X_valid_fold)
        specialist_test += specialist.predict(X_test) / viral_kf.n_splits

        fold_auc = roc_auc_score(valid_viral, probability_oof[valid_idx])
        print(f"  Fold {fold} | threshold={threshold:,.0f} | AUC={fold_auc:.4f}")

    overall_auc = roc_auc_score(actual_oof, probability_oof)
    avg_precision = average_precision_score(actual_oof, probability_oof)
    viral_store[quantile] = {
        "probability_oof": probability_oof, "specialist_oof": specialist_oof,
        "probability_test": probability_test, "specialist_test": specialist_test,
        "auc": overall_auc, "average_precision": avg_precision,
    }
    print(f"  Overall OOF AUC: {overall_auc:.4f} | Average precision: {avg_precision:.4f}")


Base model OOF RMSE: 63,101.42

Training Viral Two-Stage model | Top 10.0%
  Fold 1 | threshold=21,722 | AUC=0.9941
  Fold 2 | threshold=21,197 | AUC=0.9936
  Fold 3 | threshold=22,610 | AUC=0.9932
  Fold 4 | threshold=21,573 | AUC=0.9900
  Fold 5 | threshold=21,698 | AUC=0.9959
  Overall OOF AUC: 0.9899 | Average precision: 0.9675

Training Viral Two-Stage model | Top 5.0%
  Fold 1 | threshold=69,808 | AUC=0.9978
  Fold 2 | threshold=68,817 | AUC=0.9922
  Fold 3 | threshold=70,775 | AUC=0.9970
  Fold 4 | threshold=69,137 | AUC=0.9935
  Fold 5 | threshold=69,350 | AUC=0.9973
  Overall OOF AUC: 0.9909 | Average precision: 0.9621

Training Viral Two-Stage model | Top 2.5%
  Fold 1 | threshold=186,686 | AUC=0.9939
  Fold 2 | threshold=182,376 | AUC=0.9890
  Fold 3 | threshold=184,834 | AUC=0.9950
  Fold 4 | threshold=182,398 | AUC=0.9942
  Fold 5 | threshold=182,302 | AUC=0.9940
  Overall OOF AUC: 0.9785 | Average precision: 0.9055


### 7.1 Fine-tune the gating strength

For each candidate threshold, search over the probability exponent (`power`)
and gate strength (`gamma`) that minimize OOF RMSE.


In [102]:
fine_results = []
fine_prediction_store = {}

fine_gamma_values = np.arange(0.05, 0.251, 0.005)
fine_power_values = np.arange(1.0, 3.01, 0.10)

for quantile in viral_quantiles:
    stored = viral_store[quantile]
    probability_oof, specialist_oof = stored["probability_oof"], stored["specialist_oof"]
    probability_test, specialist_test = stored["probability_test"], stored["specialist_test"]

    correction_oof = specialist_oof - BASE_OOF
    correction_test = specialist_test - base_test_predictions

    for power in fine_power_values:
        powered_oof = probability_oof ** power
        powered_test = probability_test ** power

        for gamma in fine_gamma_values:
            candidate_oof = np.clip(BASE_OOF + gamma * powered_oof * correction_oof, 0, None)
            candidate_rmse = np.sqrt(mean_squared_error(y, candidate_oof))
            key = (quantile, round(float(power), 2), round(float(gamma), 3))

            fine_results.append({
                "viral_quantile": quantile, "power": power, "gamma": gamma, "oof_rmse": candidate_rmse
            })
            fine_prediction_store[key] = {
                "oof": candidate_oof,
                "test": np.clip(base_test_predictions + gamma * powered_test * correction_test, 0, None),
            }

fine_results = pd.DataFrame(fine_results).sort_values("oof_rmse").reset_index(drop=True)
best_fine_row = fine_results.iloc[0]

best_fine_quantile = float(best_fine_row["viral_quantile"])
best_fine_power = float(best_fine_row["power"])
best_fine_gamma = float(best_fine_row["gamma"])
best_fine_rmse = float(best_fine_row["oof_rmse"])
best_fine_key = (best_fine_quantile, round(best_fine_power, 2), round(best_fine_gamma, 3))

best_fine_oof = fine_prediction_store[best_fine_key]["oof"]
best_fine_test = fine_prediction_store[best_fine_key]["test"]

print("Selected viral configuration")
print(f"  Viral percentage : {100 * (1 - best_fine_quantile):.1f}%")
print(f"  Probability power: {best_fine_power}")
print(f"  Gamma            : {best_fine_gamma}")
print(f"  Final OOF RMSE   : {best_fine_rmse:,.2f}")


Selected viral configuration
  Viral percentage : 5.0%
  Probability power: 3.0000000000000018
  Gamma            : 0.12999999999999995
  Final OOF RMSE   : 61,409.97


## 8. Build and Save the Final Submission

Predictions are matched to `sample_submission`'s row order, validated (no
duplicates, no missing/negative/non-finite values), then saved as a CSV in
the exact format Kaggle expects.


In [103]:
final_prediction_table = pd.DataFrame({
    "video_id": test_video_ids,
    "target_day30_views": best_fine_test,
})

final_viral_submission = (
    sample_submission[["video_id"]]
    .merge(final_prediction_table, on="video_id", how="left", validate="one_to_one")
)

assert final_viral_submission.shape[0] == sample_submission.shape[0]
assert final_viral_submission["video_id"].duplicated().sum() == 0
assert final_viral_submission["target_day30_views"].isna().sum() == 0
assert (final_viral_submission["target_day30_views"] < 0).sum() == 0
assert np.isfinite(final_viral_submission["target_day30_views"]).all()

print(f"Final OOF RMSE: {best_fine_rmse:,.2f}")
display(final_viral_submission.head())

final_file_path = "/kaggle/working/viral_single_gate_rmse_61410.csv"
final_viral_submission.to_csv(final_file_path, index=False)
print("\nSaved:", final_file_path)


Final OOF RMSE: 61,409.97


,video_id,target_day30_views
0,7400582591771938090,864.414858
1,7403167206978374958,530.055333
2,7435124823820356906,2842.952627
3,7409800970143714606,735.792415
4,7410935177553333550,674.699155



Saved: /kaggle/working/viral_single_gate_rmse_61410.csv
